# 基于并行广度优先遍历的网络故障诊断——实验讲解

本实验将网络设备表示为图的顶点，将设备之间的连接表示为无向边。从一个已知故障节点出发，使用广度优先遍历（BFS）寻找距离最近的可用替代节点；若同一距离有多个候选，选择编号最小者。

希望学生通过本实验能够掌握邻接表的结构定义、头插法构建和指针遍历，理解邻接表向 CSR 连续数组的转换，说明 BFS 当前层如何分配给多个计算任务，并依据实际结果比较 CPU 串行实现与 NPU 并行实现。

本实验学习大纲如下：

1. 环境准备：创建实验目录并加载 CANN 环境；
2. 问题分析：说明邻接表、CSR、诊断规则、实验参数和对比目标；
3. 代码构建与运行：生成公共代码、Device Kernel、Host 调用、运行脚本和 CMake，完成编译与运行；
4. 正确性验证与性能分析：检查候选节点、距离与层号，比较两种执行方式的平均 BFS 时间；
5. 实验总结：归纳链式存储、CSR 转换、逐层扩展和位图合并的关系。


---
## 1. 环境准备

首先创建实验所需目录，并加载 Ascend CANN 环境变量。

目录划分如下：

- `Source/02.05/Code/include`：保存 Host 侧公共头文件。
- `Source/02.05/Code`：保存图生成、CPU 参考 BFS、计时和结果统计代码。
- `Source/02.05/ascend_ops/op_kernel`：保存 Device 侧核函数代码。
- `Source/02.05/ascend_ops/host_launch`：保存 Host 侧核函数调用代码。
- `Source/02.05/scripts`：保存构建、运行和 profiling 脚本。
- `Source/02.05/results`：保存实验结果。

需要 Python 3、Jupyter、C++17 编译器与 CMake。NPU 部分需要 CANN 和昇腾NPU；环境检测优先检查已设置的环境变量及 `/opt/conda/Ascend/cann-*`，兼容新版 tikcpp 目录与 CANN package compatibility view。兼容目录建立在工程内，不修改 CANN 安装。

下面尝试加载环境。第 3.3 节运行脚本还会检测实际 CANN 包和 `SOC_VERSION`，也可以通过 `ASCEND_INSTALL_PATH`、`SOC_VERSION` 指定。缺少 NPU 时可以先完成 CPU 路径。


In [ ]:
from pathlib import Path
import os
import subprocess

WORK_DIR = Path("Source/02.05").resolve()
for rel in ["Code/include", "ascend_ops/op_kernel", "ascend_ops/host_launch", "scripts", "results"]:
    (WORK_DIR / rel).mkdir(parents=True, exist_ok=True)
arch = os.uname().machine
bases = [Path(os.environ[k]) for k in ["ASCEND_INSTALL_PATH", "ASCEND_HOME_PATH", "ASCEND_TOOLKIT_HOME"] if os.environ.get(k)]
bases += sorted(Path("/opt/conda/Ascend").glob("cann-*"), reverse=True)
bases += [Path("/usr/local/Ascend/ascend-toolkit/latest")]
bases += sorted(Path("/usr/local/Ascend/ascend-toolkit").glob("*"), reverse=True)
set_env = None
for base in bases:
    for candidate in [base / f"{arch}-linux/set_env.sh", base / "set_env.sh"]:
        if candidate.is_file():
            set_env = candidate
            break
    if set_env:
        break
if set_env:
    raw = subprocess.check_output(["bash", "-c", 'source "$1" >&2 && env -0', "cann-env", str(set_env)])
    for item in raw.split(b"\0"):
        if b"=" in item:
            key, value = item.split(b"=", 1)
            os.environ[os.fsdecode(key)] = os.fsdecode(value)
    print("CANN environment:", set_env)
else:
    print("未找到 set_env.sh；NPU 构建时将再次检测 CANN 包。")
print("工程目录：", WORK_DIR)


---
## 2. 问题分析

### 2.1 数据结构

#### 2.1.1 邻接表

```cpp
#define MVNum 262144

typedef struct ArcNode {
    int adjvex;                 // 邻接顶点在顶点表中的位置
    struct ArcNode *nextarc;    // 指向下一条邻接边的指针
    int info;                   // 边信息，本实验中取 1，表示一跳连接
} ArcNode;

typedef struct VNode {
    int data;                   // 顶点编号，与其在顶点表中的位置相同
    ArcNode *firstarc;          // 指向该顶点的第一条邻接边
} VNode, AdjList[MVNum];

typedef struct {
    AdjList vertices;           // 顶点表
    int vexnum, arcnum;         // 实际顶点数、无向边数
} ALGraph;
```


顶点表 `vertices` 中的每个 `VNode` 保存整数 `data` 和首边指针 `firstarc`。边结点 `ArcNode` 保存邻接顶点位置 `adjvex`、后继指针 `nextarc` 和边信息 `info`。从 `vertices[u].firstarc` 出发，沿 `nextarc` 前进到空指针，就能访问顶点 u 的全部邻居。

本实验中 `vertices[i].data=i`，即顶点编号与位置相同，`adjvex` 始终保存位置。`info=1` 表示一跳连接。一条无向边建立两个边结点，`arcnum` 只增加一次。`MVNum=262144` 是容量上限，`vexnum` 是实际顶点数。

#### 2.1.2 图的 CSR 压缩存储

对图构建完邻接表后，再基于邻接表对图进行CSR压缩存储，CSR压缩存储结构为：

| 数组 | 含义 |
|---|---|
| `row_ptr` | 长度为顶点数加 1；顶点 u 的邻接段为 `[row_ptr[u], row_ptr[u+1])` |
| `col_idx` | 各顶点邻居按段连续保存；每条无向边保留两个方向 |
| `status` | 每个顶点能否作为替代节点 |

注： `row_ptr[0]=0`、`row_ptr[vexnum]=2*arcnum`。


### 2.2 实验参数

本实验的主要参数包括：

- `NODES`：图的顶点数；
- `MVNum`：顶点数的最大值，默认值为262144；
- `FAULT_NODE`：BFS 的起点编号，默认值为0；
- `BLOCK_DIM`：NPU 并行任务数，默认值为16。


### 2.3 Host/Device 侧分工

`frontier` 表示当前层等待扩展的顶点集合。初始集合只有故障节点，层号为 0。每轮执行：

1. Host 对当前frontier分块，各 NPU 任务领取连续的frontier分块；
2. Kernel 把frontier分块和邻接片段搬入 UB 中的 `LocalTensor`，在各任务的独立位图中标记发现的邻居；
3. Host 合并位图，排除已访问顶点，更新层号并生成下一层；
4. 检查本层全部新顶点，在可用顶点中选择最小编号。存在候选则结束，否则继续。

提示：找到候选后停止继续扩展，因此 `level==-1` 表示本次尚未发现，不能直接据此认定顶点不可达。


---
## 3. 代码构建与运行

本节按照实验工程的目录结构组织全部代码，并依次完成 **代码实现 → 项目构建 → 运行并输出实验结果**。

全部源码生成后，再写入根目录 `CMakeLists.txt`，最后由 `run.sh` 完成实际配置、编译和运行。`results` 保存本次运行生成的文件。


### 3.1 代码实现

代码实现部分依次生成 Host 公共头文件、Host 公共实现、Device 核函数、Host 核函数调用代码以及实验脚本：

1. `Source/02.05/Code/include`：Host 侧公共头文件；
2. `Source/02.05/Code`：图生成、CPU 参考计算、计时和结果统计代码；
3. `Source/02.05/ascend_ops/op_kernel`：Device 侧核函数代码；
4. `Source/02.05/ascend_ops/host_launch`：Host 侧核函数调用代码；
5. `Source/02.05/scripts`：构建、运行和 profiling 脚本；
6. `Source/02.05/results`：保存实验结果。


#### 3.1.1 `Source/02.05/Code/include`：Host 侧公共头文件

保存图结构、实验配置、计时、结果比较和 CPU 参考接口。


##### （1）实验参数、图结构与性能指标：`network_bfs_common.h`

先阅读 `ArcNode`、`VNode`、`AdjList` 和 `ALGraph`。`data` 保存整数编号，`adjvex` 保存邻接位置，`firstarc` 与 `nextarc` 构成单链表。`BfsConfig` 集中定义网络和执行参数，`CsrGraph` 保存转换后的连续数据，结果结构保存诊断输出与计时。


In [ ]:
%%writefile Source/02.05/Code/include/network_bfs_common.h
#pragma once

#include <algorithm>
#include <chrono>
#include <cstdint>
#include <cstdlib>
#include <iomanip>
#include <fstream>
#include <iostream>
#include <limits>
#include <numeric>
#include <memory>
#include <stdexcept>
#include <string>
#include <vector>

#define MVNum 262144
#define OK 1

namespace netbfs {

// 边结点通过 nextarc 组成单链表，adjvex 保存邻接顶点的位置。
typedef struct ArcNode {
    int adjvex;
    struct ArcNode *nextarc;
    int info;
} ArcNode;

typedef struct VNode {
    int data;
    ArcNode *firstarc;
} VNode, AdjList[MVNum];

typedef struct {
    AdjList vertices;
    int vexnum, arcnum;
} ALGraph;

constexpr int32_t kUnvisited = -1;
constexpr int32_t kNoCandidate = -1;
constexpr int32_t kDeviceInvalidCandidate = 0x7fffffff;
constexpr uint32_t kStaticMaxFrontierBlockLen = 2048;
constexpr uint32_t kStaticMaxNeighborTileLen = 2048;

struct BfsConfig {
    uint32_t num_nodes = MVNum;
    uint32_t avg_degree = 32;
    uint32_t seed = 1234;
    uint32_t fault_node = 0;
    uint32_t health_percent = 1;
    // Nodes with BFS distance <= fault_radius from fault_node are forced unavailable.
    // This creates deterministic multi-level diagnosis cases for performance tests.
    uint32_t fault_radius = 4;
    uint32_t frontier_block_len = 32;
    uint32_t neighbor_tile_len = 32;
    uint32_t block_dim = 16;
    uint32_t warmup = 2;
    uint32_t repeat = 10;
    uint32_t max_depth = 0;  // 0 means num_nodes.
    std::string csv_path;
    bool sweep = false;
    bool print_graph = false;
    bool print_levels = false;
};

struct CsrGraph {
    uint32_t num_nodes = 0;
    uint32_t undirected_edges = 0;
    std::vector<int32_t> row_ptr;
    std::vector<int32_t> col_idx;
    std::vector<int32_t> status;  // 1 means healthy and replaceable; 0 means faulty/unavailable.
};

struct TimingUs {
    double preprocess_us = 0.0;
    double bfs_total_us = 0.0;
    double kernel_us = 0.0;
    double host_frontier_us = 0.0;
};

struct BfsResult {
    int32_t replacement_node = kNoCandidate;
    int32_t distance = -1;
    uint32_t visited_count = 0;
    uint64_t processed_edges = 0;
    std::vector<int32_t> level;
    std::vector<uint32_t> frontier_sizes;
    std::vector<uint64_t> frontier_edges;
    TimingUs timing;
};

struct CheckResult {
    bool replacement_ok = false;
    bool distance_ok = false;
    bool level_ok = false;
    uint32_t level_mismatch = 0;
};

class Timer {
public:
    Timer() : start_(std::chrono::high_resolution_clock::now()) {}
    double elapsed_us() const {
        const auto end = std::chrono::high_resolution_clock::now();
        return std::chrono::duration<double, std::micro>(end - start_).count();
    }
private:
    std::chrono::high_resolution_clock::time_point start_;
};

inline uint32_t effective_max_depth(const BfsConfig& cfg) {
    return cfg.max_depth == 0 ? cfg.num_nodes : cfg.max_depth;
}

inline uint32_t round_up(uint32_t x, uint32_t align) {
    if (align == 0) return x;
    return ((x + align - 1) / align) * align;
}

inline void check_config(const BfsConfig& cfg) {
    if (cfg.num_nodes == 0 || cfg.num_nodes > MVNum) {
        throw std::invalid_argument("--nodes must be in [1, MVNum]");
    }
    if (cfg.fault_node >= cfg.num_nodes) {
        throw std::invalid_argument("--fault-node must be in [0, nodes)");
    }
    if (cfg.avg_degree == 0) {
        throw std::invalid_argument("--avg-degree must be positive");
    }
    if (cfg.health_percent > 100) {
        throw std::invalid_argument("--health-percent must be in [0, 100]");
    }
    if (cfg.frontier_block_len == 0 || cfg.frontier_block_len > kStaticMaxFrontierBlockLen) {
        throw std::invalid_argument("--frontier-block-len must be in [1, 2048]");
    }
    if (cfg.neighbor_tile_len == 0 || cfg.neighbor_tile_len > kStaticMaxNeighborTileLen) {
        throw std::invalid_argument("--neighbor-tile-len must be in [1, 2048]");
    }
    if ((cfg.frontier_block_len * sizeof(int32_t)) % 32 != 0) {
        throw std::invalid_argument("frontier_block_len*sizeof(int32_t) must be 32B aligned");
    }
    if ((cfg.neighbor_tile_len * sizeof(int32_t)) % 32 != 0) {
        throw std::invalid_argument("neighbor_tile_len*sizeof(int32_t) must be 32B aligned");
    }
    if (cfg.repeat == 0) throw std::invalid_argument("--repeat must be positive");
    if (cfg.block_dim == 0) {
        throw std::invalid_argument("--block-dim must be positive");
    }
}

inline uint64_t directed_edge_count(const CsrGraph& g) {
    return static_cast<uint64_t>(g.col_idx.size());
}

inline double edges_per_second(uint64_t edges, double us) {
    if (us <= 0.0) return 0.0;
    return static_cast<double>(edges) / (us * 1e-6);
}

inline CheckResult compare_results(const BfsResult& got, const BfsResult& ref) {
    CheckResult c;
    c.replacement_ok = (got.replacement_node == ref.replacement_node);
    c.distance_ok = (got.distance == ref.distance);
    c.level_ok = (got.level.size() == ref.level.size());
    if (c.level_ok) {
        for (size_t i = 0; i < got.level.size(); ++i) {
            if (got.level[i] != ref.level[i]) {
                ++c.level_mismatch;
                c.level_ok = false;
            }
        }
    }
    return c;
}

inline void print_graph_summary(const CsrGraph& g, uint32_t fault_node) {
    uint32_t healthy = 0;
    for (int32_t s : g.status) healthy += (s == 1 ? 1u : 0u);
    std::cout << "graph: nodes=" << g.num_nodes
              << ", directed_edges=" << g.col_idx.size()
              << ", undirected_edges=" << g.undirected_edges
              << ", healthy_nodes=" << healthy
              << ", fault_node=" << fault_node << "\n";
}

inline void print_levels_sample(const BfsResult& result, size_t count = 32) {
    const size_t n = std::min(result.level.size(), count);
    std::cout << "level sample(first " << n << "):\n";
    for (size_t i = 0; i < n; ++i) {
        std::cout << "  node=" << i << " level=" << result.level[i] << "\n";
    }
}

inline void print_result_header(bool) {
    std::cout << std::setw(14) << "method" << std::setw(16) << "avg_us"
              << std::setw(14) << "status" << '\n';
}
inline void print_result_row(const CsrGraph&, uint32_t, const BfsResult& result,
                             const std::string& status, bool ascend) {
    std::cout << std::setw(14) << (ascend ? "NPU_BFS" : "CPU_BFS")
              << std::setw(16) << std::fixed << std::setprecision(3)
              << result.timing.bfs_total_us << std::setw(14) << status << '\n';
}
inline void write_result_csv(const std::string& path, const std::string& method,
                             double avg_us, const std::string& status, bool append=false) {
    if (path.empty()) return;
    std::ofstream out(path, append ? std::ios::app : std::ios::out);
    if (!out) throw std::runtime_error("cannot write CSV: " + path);
    if (!append) out << "method,avg_us,status\n";
    out << method << ',' << std::fixed << std::setprecision(3)
        << avg_us << ',' << status << '\n';
}
} // namespace netbfs


##### （2）图生成与 CPU 参考接口：`network_bfs_cpu.h`

接口依次提供图的创建、无向边插入、链表到 CSR 的转换及释放。所有权辅助类型负责异常情况下的资源回收，结构体本身保持清晰的结点与指针定义。串行 BFS 接收 CSR 图，作为 NPU 结果的校验依据。


In [ ]:
%%writefile Source/02.05/Code/include/network_bfs_cpu.h
#pragma once

#include "network_bfs_common.h"

namespace netbfs {

// 所有权辅助代码负责异常情况下的释放；ALGraph 保持课程中的结构定义。
void destroy_graph(ALGraph* G) noexcept;
using GraphOwner = std::unique_ptr<ALGraph, decltype(&destroy_graph)>;
GraphOwner create_graph(uint32_t vertex_count);
int insert_undirected_edge(ALGraph& G, int u, int v);
CsrGraph adjacency_to_csr(const ALGraph& G, const std::vector<int32_t>& status);
CsrGraph make_network_graph(const BfsConfig& cfg);
BfsResult serial_bfs_reference(const CsrGraph& g, uint32_t fault_node, uint32_t max_depth = 0);
void dump_csr_small(const CsrGraph& g, uint32_t max_nodes = 16);

}  // namespace netbfs


#### 3.1.2 `Source/02.05/Code`：输入生成、CPU 参考计算、计时和结果统计代码

实现图生成与串行参考 BFS，并提供独立运行的 CPU 演示程序。


##### （1）网络构造与串行 BFS：`network_bfs_cpu.cpp`

构图时先建立环形骨干，再加入不重复的随机边。观察 `insert_undirected_edge`：两个边结点分配成功后，分别通过头插法连接到两个端点。`adjacency_to_csr` 沿链表计数、填写邻接数组并排序。节点状态准备完成并转换为 CSR 后，释放整个链式图。

串行 BFS 逐层扫描 CSR。首次发现邻居时标记访问状态、设置层号并加入下一层；完成整层后再确定最小编号候选。


In [ ]:
%%writefile Source/02.05/Code/network_bfs_cpu.cpp
#include "network_bfs_cpu.h"

#include <deque>
#include <random>
#include <unordered_set>

namespace netbfs {

void destroy_graph(ALGraph* G) noexcept {
    if (!G) return;
    for (int u = 0; u < G->vexnum; ++u) {
        ArcNode* p = G->vertices[u].firstarc;
        while (p != nullptr) {
            ArcNode* next = p->nextarc;
            delete p;
            p = next;
        }
        G->vertices[u].firstarc = nullptr;
    }
    G->vexnum = G->arcnum = 0;
    delete G;
}

GraphOwner create_graph(uint32_t vertex_count) {
    if (vertex_count == 0 || vertex_count > MVNum)
        throw std::invalid_argument("vertex count exceeds MVNum");
    // 顶点表放在堆上，避免大型数组占用函数栈。
    GraphOwner G(new ALGraph{}, destroy_graph);
    G->vexnum = static_cast<int>(vertex_count);
    for (int i = 0; i < G->vexnum; ++i) {
        G->vertices[i].data = i;
        G->vertices[i].firstarc = nullptr;
    }
    return G;
}

int insert_undirected_edge(ALGraph& G, int u, int v) {
    if (u < 0 || v < 0 || u >= G.vexnum || v >= G.vexnum)
        throw std::invalid_argument("edge endpoint is outside the graph");
    if (u == v) return 0;
    if (G.arcnum >= std::numeric_limits<int32_t>::max() / 2)
        throw std::overflow_error("adjacency entries exceed int32_t range");
    // 两个结点均分配成功后，再连接到图中。
    std::unique_ptr<ArcNode> forward(new ArcNode{v, nullptr, 1});
    std::unique_ptr<ArcNode> reverse(new ArcNode{u, nullptr, 1});
    forward->nextarc = G.vertices[u].firstarc;
    G.vertices[u].firstarc = forward.release();
    reverse->nextarc = G.vertices[v].firstarc;
    G.vertices[v].firstarc = reverse.release();
    ++G.arcnum;
    return OK;
}

CsrGraph adjacency_to_csr(const ALGraph& G, const std::vector<int32_t>& status) {
    if (G.vexnum <= 0 || G.vexnum > MVNum || G.arcnum < 0 ||
        status.size() != static_cast<size_t>(G.vexnum))
        throw std::invalid_argument("invalid ALGraph or vertex status");
    CsrGraph g;
    g.num_nodes = static_cast<uint32_t>(G.vexnum);
    g.undirected_edges = static_cast<uint32_t>(G.arcnum);
    g.row_ptr.assign(static_cast<size_t>(G.vexnum) + 1, 0);
    uint64_t entries = 0;
    // 第一遍：沿链表计数，以累计长度记录各邻接段边界。
    for (int u = 0; u < G.vexnum; ++u) {
        if (G.vertices[u].data != u || (status[u] != 0 && status[u] != 1))
            throw std::invalid_argument("invalid vertex data or status");
        for (const ArcNode* p = G.vertices[u].firstarc; p != nullptr; p = p->nextarc) {
            if (p->adjvex < 0 || p->adjvex >= G.vexnum || p->info != 1)
                throw std::invalid_argument("invalid adjacency node");
            if (++entries > static_cast<uint64_t>(G.arcnum) * 2 ||
                entries > static_cast<uint64_t>(std::numeric_limits<int32_t>::max()))
                throw std::runtime_error("adjacency count exceeds graph metadata");
        }
        g.row_ptr[u + 1] = static_cast<int32_t>(entries);
    }
    if (entries != static_cast<uint64_t>(G.arcnum) * 2)
        throw std::runtime_error("each undirected edge must have two adjacency nodes");
    // 第二遍：将边结点中的邻接位置写入连续数组。
    g.col_idx.resize(static_cast<size_t>(entries));
    for (int u = 0; u < G.vexnum; ++u) {
        int32_t position = g.row_ptr[u];
        for (const ArcNode* p = G.vertices[u].firstarc; p != nullptr; p = p->nextarc)
            g.col_idx[position++] = p->adjvex;
        std::sort(g.col_idx.begin() + g.row_ptr[u], g.col_idx.begin() + g.row_ptr[u + 1]);
    }
    g.status = status;
    return g;
}

namespace {
uint64_t edge_key(uint32_t u, uint32_t v) {
    if (u > v) std::swap(u, v);
    return (static_cast<uint64_t>(u) << 32) | static_cast<uint64_t>(v);
}

void add_unique_edge(uint32_t u, uint32_t v, ALGraph& G,
                     std::unordered_set<uint64_t>& used) {
    if (u == v || !used.insert(edge_key(u, v)).second) return;
    insert_undirected_edge(G, static_cast<int>(u), static_cast<int>(v));
}

// 用于准备节点状态，距离只计算到给定半径。
std::vector<int32_t> collect_nodes_within_radius(const ALGraph& G,
                                                uint32_t source, uint32_t radius) {
    std::vector<int32_t> depth(G.vexnum, -1);
    std::deque<uint32_t> q;
    depth[source] = 0;
    q.push_back(source);
    while (!q.empty()) {
        const uint32_t u = q.front();
        q.pop_front();
        if (static_cast<uint32_t>(depth[u]) >= radius) continue;
        for (const ArcNode* p = G.vertices[u].firstarc; p != nullptr; p = p->nextarc) {
            const uint32_t v = static_cast<uint32_t>(p->adjvex);
            if (depth[v] >= 0) continue;
            depth[v] = depth[u] + 1;
            q.push_back(v);
        }
    }
    return depth;
}
}  // namespace

CsrGraph make_network_graph(const BfsConfig& cfg) {
    check_config(cfg);
    const uint32_t n = cfg.num_nodes;
    auto G = create_graph(n);
    std::unordered_set<uint64_t> used;
    used.reserve(static_cast<size_t>(n) * std::max(2u, cfg.avg_degree));
    if (n >= 2) {
        for (uint32_t u = 0; u < n; ++u)
            add_unique_edge(u, (u + 1) % n, *G, used);
    }
    const uint64_t target = std::max<uint64_t>(used.size(),
        std::max<uint64_t>(1, static_cast<uint64_t>(n) * cfg.avg_degree / 2));
    std::mt19937 rng(cfg.seed);
    std::uniform_int_distribution<uint32_t> node_dist(0, n - 1);
    uint64_t guard = 0;
    const uint64_t guard_limit = target * 20 + 1024;
    while (used.size() < target && guard++ < guard_limit) {
        const uint32_t u = node_dist(rng);
        uint32_t v = node_dist(rng);
        if (u == v) v = (v + 1) % n;
        add_unique_edge(u, v, *G, used);
    }
    std::vector<int32_t> status(n, 0);
    std::uniform_int_distribution<uint32_t> pct_dist(1, 100);
    for (uint32_t u = 0; u < n; ++u) status[u] = pct_dist(rng) <= cfg.health_percent ? 1 : 0;
    std::vector<int32_t> near_fault;
    if (cfg.fault_radius > 0) {
        near_fault = collect_nodes_within_radius(*G, cfg.fault_node, cfg.fault_radius);
        for (uint32_t u = 0; u < n; ++u)
            if (near_fault[u] >= 0) status[u] = 0;
    } else {
        status[cfg.fault_node] = 0;
    }
    if (std::find(status.begin(), status.end(), 1) == status.end() && n > 1) {
        if (near_fault.empty())
            near_fault = collect_nodes_within_radius(*G, cfg.fault_node, cfg.fault_radius);
        for (uint32_t step = 1; step < n; ++step) {
            const uint32_t candidate = (cfg.fault_node + step) % n;
            if (candidate != cfg.fault_node && (cfg.fault_radius == 0 || near_fault[candidate] < 0)) {
                status[candidate] = 1;
                break;
            }
        }
    }
    auto graph = adjacency_to_csr(*G, status);
    G.reset();  // CSR 转换完成，释放全部链表结点和顶点表。
    return graph;
}

BfsResult serial_bfs_reference(const CsrGraph& g, uint32_t fault_node, uint32_t max_depth) {
    if (fault_node >= g.num_nodes) {
        throw std::invalid_argument("fault_node must be in [0, num_nodes)");
    }
    if (g.row_ptr.size() != static_cast<size_t>(g.num_nodes) + 1 ||
        g.status.size() != g.num_nodes) {
        throw std::invalid_argument("invalid CSR graph");
    }
    const uint32_t depth_limit = max_depth == 0 ? g.num_nodes : max_depth;

    BfsResult result;
    result.level.assign(g.num_nodes, kUnvisited);
    std::vector<int32_t> visited(g.num_nodes, 0);
    std::vector<int32_t> frontier;
    frontier.reserve(g.num_nodes);
    frontier.push_back(static_cast<int32_t>(fault_node));
    visited[fault_node] = 1;
    result.level[fault_node] = 0;
    result.visited_count = 1;

    Timer timer;
    for (uint32_t depth = 0; !frontier.empty() && depth < depth_limit; ++depth) {
        result.frontier_sizes.push_back(static_cast<uint32_t>(frontier.size()));
        uint64_t edges_this_level = 0;
        std::vector<int32_t> next;
        next.reserve(frontier.size() * 2);
        int32_t best_candidate = kDeviceInvalidCandidate;

        for (int32_t u : frontier) {
            if (u < 0 || static_cast<uint32_t>(u) >= g.num_nodes) continue;
            const int32_t begin = g.row_ptr[static_cast<size_t>(u)];
            const int32_t end = g.row_ptr[static_cast<size_t>(u) + 1];
            edges_this_level += static_cast<uint64_t>(std::max(0, end - begin));
            for (int32_t p = begin; p < end; ++p) {
                const int32_t v = g.col_idx[static_cast<size_t>(p)];
                if (v < 0 || static_cast<uint32_t>(v) >= g.num_nodes) continue;
                if (visited[static_cast<size_t>(v)] == 0) {
                    visited[static_cast<size_t>(v)] = 1;
                    result.level[static_cast<size_t>(v)] = static_cast<int32_t>(depth + 1);
                    ++result.visited_count;
                    next.push_back(v);
                    if (g.status[static_cast<size_t>(v)] == 1 && v < best_candidate) {
                        best_candidate = v;
                    }
                }
            }
        }

        result.processed_edges += edges_this_level;
        result.frontier_edges.push_back(edges_this_level);
        if (best_candidate != kDeviceInvalidCandidate) {
            result.replacement_node = best_candidate;
            result.distance = static_cast<int32_t>(depth + 1);
            break;
        }
        frontier.swap(next);
    }
    result.timing.bfs_total_us = timer.elapsed_us();
    return result;
}

void dump_csr_small(const CsrGraph& g, uint32_t max_nodes) {
    const uint32_t n = std::min(g.num_nodes, max_nodes);
    std::cout << "CSR sample(first " << n << " nodes):\n";
    for (uint32_t u = 0; u < n; ++u) {
        std::cout << "  node=" << u << " status=" << g.status[u] << " neighbors=[";
        const int32_t begin = g.row_ptr[u];
        const int32_t end = g.row_ptr[u + 1];
        for (int32_t p = begin; p < end; ++p) {
            if (p > begin) std::cout << ",";
            std::cout << g.col_idx[static_cast<size_t>(p)];
        }
        std::cout << "]\n";
    }
}

}  // namespace netbfs


##### （2）CPU 演示程序：`main.cpp`

主程序读取参数并完成链式构图、状态准备和 CSR 转换。预处理时间包含链表释放；正式计时使用统一的预热与重复次数，平均值只覆盖 CSR 串行 BFS。


In [ ]:
%%writefile Source/02.05/Code/main.cpp
#include "network_bfs_cpu.h"

#include <cstring>

namespace {

void usage(const char* argv0) {
    std::cout << "Usage: " << argv0 << " [options]\n"
              << "Options:\n"
              << "  --nodes <num>                node count, default: 262144\n"
              << "  --avg-degree <num>           target average undirected degree, default: 32\n"
              << "  --fault-node <id>            source faulty node, default: 0\n"
              << "  --health-percent <0..100>    probability of a node being healthy, default: 1\n"
              << "  --fault-radius <num>         force nodes within this BFS radius unavailable, default: 4\n"
              << "  --seed <num>                 random seed, default: 1234\n"
              << "  --max-depth <num>            BFS depth limit, default: nodes\n"
              << "  --warmup / --repeat / --csv   timing counts and output CSV\n"
              << "  --print-graph                print a small CSR sample\n"
              << "  --print-levels               print first 32 levels\n"
              << "  -h, --help                   show help\n";
}

netbfs::BfsConfig parse_args(int argc, char** argv) {
    netbfs::BfsConfig cfg;
    for (int i = 1; i < argc; ++i) {
        const std::string arg = argv[i];
        auto need_value = [&](const std::string& name) -> const char* {
            if (i + 1 >= argc) throw std::invalid_argument("missing value after " + name);
            return argv[++i];
        };
        if (arg == "--nodes") {
            cfg.num_nodes = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--avg-degree") {
            cfg.avg_degree = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--fault-node") {
            cfg.fault_node = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--health-percent") {
            cfg.health_percent = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--fault-radius") {
            cfg.fault_radius = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--seed") {
            cfg.seed = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--max-depth") {
            cfg.max_depth = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--warmup") {
            cfg.warmup = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--repeat") {
            cfg.repeat = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--csv") {
            cfg.csv_path = need_value(arg);
        } else if (arg == "--print-graph") {
            cfg.print_graph = true;
        } else if (arg == "--print-levels") {
            cfg.print_levels = true;
        } else if (arg == "-h" || arg == "--help") {
            usage(argv[0]);
            std::exit(0);
        } else {
            throw std::invalid_argument("unknown argument: " + arg);
        }
    }
    return cfg;
}

}  // namespace

int main(int argc, char** argv) {
    try {
        const auto cfg = parse_args(argc, argv);
        netbfs::check_config(cfg);

        netbfs::Timer prep_timer;
        auto graph = netbfs::make_network_graph(cfg);
        const double prep_us = prep_timer.elapsed_us();
        if (cfg.print_graph) netbfs::dump_csr_small(graph);
        netbfs::print_graph_summary(graph, cfg.fault_node);

        for (uint32_t i = 0; i < cfg.warmup; ++i)
            (void)netbfs::serial_bfs_reference(graph, cfg.fault_node, netbfs::effective_max_depth(cfg));
        netbfs::BfsResult ref;
        double total = 0.0;
        for (uint32_t i = 0; i < cfg.repeat; ++i) {
            ref = netbfs::serial_bfs_reference(graph, cfg.fault_node, netbfs::effective_max_depth(cfg));
            total += ref.timing.bfs_total_us;
        }
        ref.timing.bfs_total_us = total / cfg.repeat;
        ref.timing.preprocess_us = prep_us;

        std::cout << "preprocess_us=" << prep_us << "\n";
        netbfs::print_result_header(false);
        netbfs::print_result_row(graph, cfg.fault_node, ref, "REFERENCE", false);
        netbfs::write_result_csv(cfg.csv_path, "CPU_BFS", ref.timing.bfs_total_us, "REFERENCE");
        std::cout << "diagnosis: replacement=" << ref.replacement_node
                  << ", distance=" << ref.distance << '\n';
        std::cout << "frontier sizes:";
        for (size_t i = 0; i < ref.frontier_sizes.size(); ++i) {
            std::cout << " L" << i << "=" << ref.frontier_sizes[i]
                      << "(E=" << ref.frontier_edges[i] << ")";
        }
        std::cout << "\n";
        if (cfg.print_levels) netbfs::print_levels_sample(ref);
        return 0;
    } catch (const std::exception& e) {
        std::cerr << "error: " << e.what() << "\n";
        usage(argv[0]);
        return 1;
    }
}


#### 3.1.3 `Source/02.05/ascend_ops/op_kernel`：Device 侧核函数代码

每次启动只扩展一层前沿。遍历是否结束、访问状态更新与候选选择由 Host 控制。


##### （1）单层邻居扩展 Kernel：`bfs_expand_level.cpp`

将前沿分配到不同任务，使用静态 LocalTensor 分段读取邻居，在各自位图中记录发现结果。位图按缓存行隔离，写回后刷新缓存。


In [ ]:
%%writefile Source/02.05/ascend_ops/op_kernel/bfs_expand_level.cpp
// Experiment 2 kernel: one BFS level expansion over a CSR graph.
//
// Host-merged per-core bitset design:
//   - current frontier is stored as a dense int32 array in GM;
//   - CSR row_ptr and col_idx encode adjacency lists;
//   - each AI Core owns a contiguous range of frontier blocks;
//   - static UB tensors stage frontier nodes and neighbor segments;
//   - the kernel writes a per-core bitset slice for discovered neighbor
//     vertices, using one bit per graph vertex;
//   - Host OR-reduces the per-core bitsets and performs the canonical
//     visited/level update.
//
// Private per-core bitsets provide race-free discovery while reducing the
// returned next-frontier state from blockDim*numNodes int32 flags to
// blockDim*round_up(ceil(numNodes/32),16) int32 bitset words.

#include "kernel_operator.h"

using namespace AscendC;

namespace {
constexpr uint32_t kStaticMaxFrontierBlockLen = 2048;
constexpr uint32_t kStaticMaxNeighborTileLen = 2048;

__aicore__ inline void GetBlockRange(uint32_t numBlocks,
                                      uint32_t launchBlockDim,
                                      uint32_t& beginBlock,
                                      uint32_t& endBlock) {
    const uint32_t coreNum = launchBlockDim == 0 ? 1 : launchBlockDim;
    const uint32_t coreId = GetBlockIdx();
    if (coreId >= coreNum) {
        beginBlock = 0;
        endBlock = 0;
        return;
    }
    const uint32_t blocksPerCore = (numBlocks + coreNum - 1) / coreNum;
    beginBlock = coreId * blocksPerCore;
    endBlock = beginBlock + blocksPerCore;
    if (endBlock > numBlocks) {
        endBlock = numBlocks;
    }
}

__aicore__ inline uint32_t MinU32(uint32_t a, uint32_t b) {
    return a < b ? a : b;
}
}  // namespace

extern "C" __global__ __aicore__ void bfs_expand_level(GM_ADDR rowPtr,
                                                        GM_ADDR colIdx,
                                                        GM_ADDR frontier,
                                                        GM_ADDR nextBitmap,
                                                        uint32_t numNodes,
                                                        uint32_t numEdgesPadded,
                                                        uint32_t frontierCount,
                                                        uint32_t frontierPaddedCount,
                                                        uint32_t frontierBlockLen,
                                                        uint32_t neighborTileLen,
                                                        uint32_t launchBlockDim) {
    InitSocState();

    GlobalTensor<int32_t> rowPtrGm;
    GlobalTensor<int32_t> colIdxGm;
    GlobalTensor<int32_t> frontierGm;
    GlobalTensor<int32_t> nextBitmapGm;

    rowPtrGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t*>(rowPtr), numNodes + 1);
    colIdxGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t*>(colIdx), numEdgesPadded);
    frontierGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t*>(frontier), frontierPaddedCount);
    const uint32_t wordsPerBitmap = (((numNodes + 31u) >> 5) + 15u) / 16u * 16u;
    nextBitmapGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t*>(nextBitmap), wordsPerBitmap * launchBlockDim);

    const uint32_t coreId = GetBlockIdx();
    if (coreId >= launchBlockDim) {
        return;
    }

    if (frontierBlockLen == 0 || frontierBlockLen > kStaticMaxFrontierBlockLen ||
        neighborTileLen == 0 || neighborTileLen > kStaticMaxNeighborTileLen ||
        frontierCount == 0 || numNodes == 0 || numEdgesPadded == 0) {
        return;
    }

    const uint32_t numFrontierBlocks = (frontierCount + frontierBlockLen - 1) / frontierBlockLen;
    uint32_t beginBlock = 0;
    uint32_t endBlock = 0;
    GetBlockRange(numFrontierBlocks, launchBlockDim, beginBlock, endBlock);
    if (beginBlock >= endBlock) {
        return;
    }

    LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    LocalTensor<int32_t> frontierLocal = ubAllocator.Alloc<int32_t, kStaticMaxFrontierBlockLen>();
    LocalTensor<int32_t> neighborLocal = ubAllocator.Alloc<int32_t, kStaticMaxNeighborTileLen>();
    frontierLocal.SetSize(kStaticMaxFrontierBlockLen);
    neighborLocal.SetSize(kStaticMaxNeighborTileLen);


    for (uint32_t blockId = beginBlock; blockId < endBlock; ++blockId) {
        const uint32_t frontierBase = blockId * frontierBlockLen;
        const uint32_t activeFrontierLen = MinU32(frontierBlockLen, frontierCount - frontierBase);
        DataCopy(frontierLocal, frontierGm[frontierBase], frontierBlockLen);
        PipeBarrier<PIPE_ALL>();

        for (uint32_t i = 0; i < activeFrontierLen; ++i) {
            const int32_t u = frontierLocal.GetValue(i);
            if (u < 0 || static_cast<uint32_t>(u) >= numNodes) {
                continue;
            }
            const int32_t begin = rowPtrGm.GetValue(static_cast<uint32_t>(u));
            const int32_t end = rowPtrGm.GetValue(static_cast<uint32_t>(u) + 1);
            if (end <= begin) {
                continue;
            }

            uint32_t edgePos = static_cast<uint32_t>(begin);
            const uint32_t edgeEnd = static_cast<uint32_t>(end);
            while (edgePos < edgeEnd) {
                const uint32_t activeNeighborLen = MinU32(neighborTileLen, edgeEnd - edgePos);
                DataCopy(neighborLocal, colIdxGm[edgePos], neighborTileLen);
                PipeBarrier<PIPE_ALL>();

                for (uint32_t j = 0; j < activeNeighborLen; ++j) {
                    const int32_t v = neighborLocal.GetValue(j);
                    if (v < 0 || static_cast<uint32_t>(v) >= numNodes) {
                        continue;
                    }
                    // Per-core bitset layout:
                    // nextBitmap[coreId * wordsPerBitmap + (vertex / 32)].
                    // Each AI Core owns a private slice, so there is no cross-core
                    // write race. Host later OR-reduces these slices and assigns the
                    // canonical BFS level for the next frontier.
                    const uint32_t vu = static_cast<uint32_t>(v);
                    const uint32_t wordOffset = vu >> 5;
                    const uint32_t bitMask = 1u << (vu & 31u);
                    const uint32_t bitmapIndex = coreId * wordsPerBitmap + wordOffset;
                    const uint32_t oldWord = static_cast<uint32_t>(nextBitmapGm.GetValue(bitmapIndex));
                    nextBitmapGm.SetValue(bitmapIndex, static_cast<int32_t>(oldWord | bitMask));
                }
                edgePos += activeNeighborLen;
            }
        }
    }

    // 将本核位图写回 GM，并使后续层不再使用旧缓存。
    DataCacheCleanAndInvalid<int32_t, CacheLine::ENTIRE_DATA_CACHE,
                             DcciDst::CACHELINE_OUT>(nextBitmapGm);
}


#### 3.1.4 `Source/02.05/ascend_ops/host_launch`：Host 侧核函数调用代码

Host 负责数据传输、逐层启动、位图合并、访问状态更新、候选选择和验证。下面按四部分依次写入同一文件：第一段覆盖写入，后三段追加。


##### （1）基础定义与运行参数结构

写入 ACL 错误检查、配置结构和运行参数说明。


In [ ]:
%%writefile Source/02.05/ascend_ops/host_launch/bfs_npu_main.cpp
#include <acl/acl.h>
#include <aclrtlaunch_bfs_expand_level.h>

#include "network_bfs_cpu.h"

#include <cstdint>
#include <cstdlib>
#include <dlfcn.h>
#include <iomanip>
#include <iostream>
#include <stdexcept>
#include <string>
#include <vector>

#define ACL_CHECK(expr)                                                              \
    do {                                                                             \
        const aclError _ret = (expr);                                                 \
        if (_ret != ACL_SUCCESS) {                                                    \
            std::string _message = std::string("ACL error: ") + #expr +                \
                                   ", code=" + std::to_string(static_cast<int>(_ret)); \
            const char* _detail = aclGetRecentErrMsg();                                \
            if (_detail && *_detail) _message += std::string("\n") + _detail;           \
            if (_ret == ACL_ERROR_RT_DEV_SETUP_ERROR) {                               \
                _message += "\nDevice startup failed. Check npu-smi info, the "       \
                            "assigned device mapping, and the CANN/driver runtime."; \
            }                                                                        \
            throw std::runtime_error(_message);                                       \
        }                                                                            \
    } while (0)

namespace {

void report_device_environment() {
    Dl_info library{};
    if (dladdr(reinterpret_cast<const void*>(&aclInit), &library) != 0 && library.dli_fname) {
        std::cout << "[INFO] AscendCL library=" << library.dli_fname << '\n';
        if (std::string(library.dli_fname).find("/stub/") != std::string::npos) {
            throw std::runtime_error("A link-time stub library was loaded. Remove stub paths "
                                     "from LD_LIBRARY_PATH and rebuild against the CANN runtime.");
        }
    }
    for (const char* name : {"ASCEND_RT_VISIBLE_DEVICES", "ASCEND_VISIBLE_DEVICES"}) {
        const char* value = std::getenv(name);
        std::cout << "[INFO] " << name << '=' << (value ? value : "<unset>") << '\n';
    }
    std::cout << std::flush;
}

void validate_device_index(int32_t device) {
    uint32_t count = 0;
    ACL_CHECK(aclrtGetDeviceCount(&count));
    std::cout << "[INFO] ACL visible device count=" << count
              << ", selected device index=" << device << std::endl;
    if (count == 0) {
        throw std::runtime_error("No NPU device is visible to this process. Check the "
                                 "Notebook resource allocation and container device mapping.");
    }
    if (device < 0 || static_cast<uint32_t>(device) >= count) {
        throw std::runtime_error("--device must be a visible device index in [0, " +
                                 std::to_string(count - 1) + "]. Check the assigned device "
                                 "mapping before selecting an index.");
    }
}

struct Config : public netbfs::BfsConfig {
    int32_t device = 0;
    bool check_device_only = false;
};

void usage(const char* argv0) {
    std::cout << "Usage: " << argv0 << " [options]\n"
              << "Options:\n"
              << "  --device <id>                visible device index, default: 0\n"
              << "  --check-device               check device startup and exit\n"
              << "  --nodes <num>                node count, default: 262144\n"
              << "  --avg-degree <num>           target average undirected degree, default: 32\n"
              << "  --fault-node <id>            source faulty node, default: 0\n"
              << "  --health-percent <0..100>    probability of healthy nodes, default: 1\n"
              << "  --fault-radius <num>         force nodes within this BFS radius unavailable, default: 4\n"
              << "  --frontier-block-len <num>   static Tensor frontier block length, default: 32\n"
              << "  --neighbor-tile-len <num>    static Tensor neighbor tile length, default: 32\n"
              << "  --block-dim <num>            AI Core launch blockDim, default: 16\n"
              << "  --warmup <num>               warmup count, default: 2\n"
              << "  --repeat <num>               repeat count, default: 10\n"
              << "  --seed <num>                 random seed, default: 1234\n"
              << "  --max-depth <num>            BFS depth limit, default: nodes\n"
              << "  --csv <path>                 result CSV\n"
              << "  --sweep                      test frontier block len 256/512/1024/2048\n"
              << "  --print-graph                print a small CSR sample\n"
              << "  --print-levels               print first 32 output levels\n"
              << "  -h, --help                   show help\n";
}


##### （2）参数解析与设备缓冲区

解析参数，补齐输入并定义设备内存的申请与释放接口。


In [ ]:
%%writefile -a Source/02.05/ascend_ops/host_launch/bfs_npu_main.cpp
Config parse_args(int argc, char** argv) {
    Config cfg;
    for (int i = 1; i < argc; ++i) {
        const std::string arg = argv[i];
        auto need_value = [&](const std::string& name) -> const char* {
            if (i + 1 >= argc) throw std::invalid_argument("missing value after " + name);
            return argv[++i];
        };
        if (arg == "--device") {
            cfg.device = std::stoi(need_value(arg));
        } else if (arg == "--check-device") {
            cfg.check_device_only = true;
        } else if (arg == "--nodes") {
            cfg.num_nodes = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--avg-degree") {
            cfg.avg_degree = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--fault-node") {
            cfg.fault_node = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--health-percent") {
            cfg.health_percent = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--fault-radius") {
            cfg.fault_radius = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--frontier-block-len") {
            cfg.frontier_block_len = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--neighbor-tile-len") {
            cfg.neighbor_tile_len = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--block-dim") {
            cfg.block_dim = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--warmup") {
            cfg.warmup = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--repeat") {
            cfg.repeat = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--seed") {
            cfg.seed = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--max-depth") {
            cfg.max_depth = static_cast<uint32_t>(std::stoul(need_value(arg)));
        } else if (arg == "--csv") {
            cfg.csv_path = need_value(arg);
        } else if (arg == "--sweep") {
            cfg.sweep = true;
        } else if (arg == "--print-graph") {
            cfg.print_graph = true;
        } else if (arg == "--print-levels") {
            cfg.print_levels = true;
        } else if (arg == "-h" || arg == "--help") {
            usage(argv[0]);
            std::exit(0);
        } else {
            throw std::invalid_argument("unknown argument: " + arg);
        }
    }
    return cfg;
}

std::vector<int32_t> make_padded_colidx(const netbfs::CsrGraph& g, uint32_t neighborTileLen) {
    const uint32_t padded = netbfs::round_up(static_cast<uint32_t>(g.col_idx.size()) + neighborTileLen, neighborTileLen);
    std::vector<int32_t> out(padded, -1);
    std::copy(g.col_idx.begin(), g.col_idx.end(), out.begin());
    return out;
}

std::vector<int32_t> make_padded_frontier(const std::vector<int32_t>& frontier, uint32_t capacity, uint32_t blockLen) {
    const uint32_t active = static_cast<uint32_t>(frontier.size());
    const uint32_t padded = std::max<uint32_t>(blockLen, netbfs::round_up(std::max(active, 1u), blockLen));
    if (padded > capacity) {
        throw std::runtime_error("frontier capacity exceeded");
    }
    std::vector<int32_t> out(padded, -1);
    std::copy(frontier.begin(), frontier.end(), out.begin());
    return out;
}

uint32_t count_visited(const std::vector<int32_t>& level) {
    uint32_t n = 0;
    for (int32_t v : level) n += (v >= 0 ? 1u : 0u);
    return n;
}

bool is_valid_device_candidate(const netbfs::CsrGraph& graph, uint32_t faultNode, int32_t v) {
    if (v < 0) return false;
    const uint32_t vu = static_cast<uint32_t>(v);
    if (vu >= graph.num_nodes) return false;
    if (vu == faultNode) return false;
    return graph.status[vu] == 1;
}

struct DeviceBuffers {
    void* rowPtr = nullptr;
    void* colIdx = nullptr;
    void* frontier = nullptr;
    void* nextBitmap = nullptr;
};

void free_buffers(DeviceBuffers& b) {
    if (b.nextBitmap) aclrtFree(b.nextBitmap);
    if (b.frontier) aclrtFree(b.frontier);
    if (b.colIdx) aclrtFree(b.colIdx);
    if (b.rowPtr) aclrtFree(b.rowPtr);
    b = DeviceBuffers{};
}


##### （3）逐层 Kernel 启动与计时

扩展当前层，合并各任务位图并构造下一层，重复运行并记录时间。


In [ ]:
%%writefile -a Source/02.05/ascend_ops/host_launch/bfs_npu_main.cpp
netbfs::BfsResult run_device_bfs_once(const Config& cfg,
                                    const netbfs::CsrGraph& graph,
                                    const std::vector<int32_t>& colIdxPadded,
                                    DeviceBuffers& dev,
                                    aclrtStream stream,
                                    bool timed) {
    const uint32_t n = graph.num_nodes;
    const uint32_t numEdgesPadded = static_cast<uint32_t>(colIdxPadded.size());
    const uint32_t frontierCapacity = netbfs::round_up(n + cfg.frontier_block_len, cfg.frontier_block_len);
    const size_t frontierBytes = static_cast<size_t>(frontierCapacity) * sizeof(int32_t);
    const uint32_t wordsPerBitmap = (((n + 31u) >> 5) + 15u) / 16u * 16u;
    const size_t nextBitmapBytes = static_cast<size_t>(cfg.block_dim) * wordsPerBitmap * sizeof(uint32_t);

    std::vector<int32_t> visitedHost(n, 0);
    std::vector<int32_t> levelHost(n, netbfs::kUnvisited);
    visitedHost[cfg.fault_node] = 1;
    levelHost[cfg.fault_node] = 0;

    // Host keeps visited/level as authoritative state. The Device expands
    // the current frontier and writes per-core compact bitsets. This keeps the
    // race-free layer semantics of per-core discovery while reducing transfer
    // volume by 32x compared with int32 per-core bitmaps.

    netbfs::BfsResult result;
    std::vector<int32_t> frontier{static_cast<int32_t>(cfg.fault_node)};
    std::vector<uint32_t> nextBitmap(static_cast<size_t>(cfg.block_dim) * wordsPerBitmap, 0);
    std::vector<uint32_t> mergedBitmap(wordsPerBitmap, 0);

    netbfs::Timer totalTimer;
    double kernelUs = 0.0;
    double frontierUs = 0.0;
    const uint32_t depthLimit = netbfs::effective_max_depth(cfg);

    for (uint32_t depth = 0; !frontier.empty() && depth < depthLimit; ++depth) {
        result.frontier_sizes.push_back(static_cast<uint32_t>(frontier.size()));

        netbfs::Timer frontierTimer;
        auto paddedFrontier = make_padded_frontier(frontier, frontierCapacity, cfg.frontier_block_len);
        const uint32_t frontierPaddedCount = static_cast<uint32_t>(paddedFrontier.size());
        const size_t paddedFrontierBytes = static_cast<size_t>(frontierPaddedCount) * sizeof(int32_t);
        ACL_CHECK(aclrtMemcpy(dev.frontier, frontierBytes, paddedFrontier.data(), paddedFrontierBytes, ACL_MEMCPY_HOST_TO_DEVICE));
        ACL_CHECK(aclrtMemset(dev.nextBitmap, nextBitmapBytes, 0, nextBitmapBytes));
        if (timed) frontierUs += frontierTimer.elapsed_us();

        netbfs::Timer kernelTimer;
        ACLRT_LAUNCH_KERNEL(bfs_expand_level)(cfg.block_dim, stream,
                                              dev.rowPtr, dev.colIdx, dev.frontier, dev.nextBitmap,
                                              n, numEdgesPadded,
                                              static_cast<uint32_t>(frontier.size()), frontierPaddedCount,
                                              cfg.frontier_block_len, cfg.neighbor_tile_len,
                                              cfg.block_dim);
        ACL_CHECK(aclrtSynchronizeStream(stream));
        if (timed) kernelUs += kernelTimer.elapsed_us();

        netbfs::Timer postTimer;

        uint64_t edgesThisLevel = 0;
        for (int32_t u : frontier) {
            if (u < 0 || static_cast<uint32_t>(u) >= n) continue;
            const int32_t begin = graph.row_ptr[static_cast<size_t>(u)];
            const int32_t end = graph.row_ptr[static_cast<size_t>(u) + 1];
            if (end > begin) edgesThisLevel += static_cast<uint64_t>(end - begin);
        }
        result.frontier_edges.push_back(edgesThisLevel);
        result.processed_edges += edgesThisLevel;

        ACL_CHECK(aclrtMemcpy(nextBitmap.data(), nextBitmapBytes, dev.nextBitmap,
                              nextBitmapBytes, ACL_MEMCPY_DEVICE_TO_HOST));

        std::fill(mergedBitmap.begin(), mergedBitmap.end(), 0u);
        for (uint32_t c = 0; c < cfg.block_dim; ++c) {
            const size_t base = static_cast<size_t>(c) * wordsPerBitmap;
            for (uint32_t w = 0; w < wordsPerBitmap; ++w) {
                mergedBitmap[w] |= nextBitmap[base + w];
            }
        }

        int32_t best = netbfs::kDeviceInvalidCandidate;
        std::vector<int32_t> next;
        next.reserve(std::min<size_t>(n, std::max<size_t>(frontier.size() * 2, 64)));
        for (uint32_t w = 0; w < wordsPerBitmap; ++w) {
            uint32_t word = mergedBitmap[w];
            while (word != 0u) {
                const uint32_t bit = static_cast<uint32_t>(__builtin_ctz(word));
                const uint32_t v = (w << 5) + bit;
                word &= (word - 1u);
                if (v >= n || visitedHost[v] != 0) {
                    continue;
                }
                visitedHost[v] = 1;
                levelHost[v] = static_cast<int32_t>(depth + 1);
                next.push_back(static_cast<int32_t>(v));
                if (is_valid_device_candidate(graph, cfg.fault_node, static_cast<int32_t>(v)) &&
                    static_cast<int32_t>(v) < best) {
                    best = static_cast<int32_t>(v);
                }
            }
        }

        if (best != netbfs::kDeviceInvalidCandidate) {
            result.replacement_node = best;
            result.distance = static_cast<int32_t>(depth + 1);
            if (timed) frontierUs += postTimer.elapsed_us();
            break;
        }

        frontier.swap(next);
        if (timed) frontierUs += postTimer.elapsed_us();
    }

    result.timing.bfs_total_us = timed ? totalTimer.elapsed_us() : 0.0;
    result.timing.kernel_us = kernelUs;
    result.timing.host_frontier_us = frontierUs;
    result.level = levelHost;
    result.visited_count = count_visited(result.level);

    return result;
}

bool run_one(const Config& cfg, const netbfs::CsrGraph& graph, const netbfs::BfsResult& ref, bool append=false) {
    netbfs::check_config(cfg);

    const uint32_t n = graph.num_nodes;
    const auto colIdxPadded = make_padded_colidx(graph, cfg.neighbor_tile_len);
    const uint32_t frontierCapacity = netbfs::round_up(n + cfg.frontier_block_len, cfg.frontier_block_len);

    DeviceBuffers dev;
    aclrtStream stream = nullptr;

    const size_t rowPtrBytes = static_cast<size_t>(n + 1) * sizeof(int32_t);
    const size_t colIdxBytes = static_cast<size_t>(colIdxPadded.size()) * sizeof(int32_t);
    const size_t frontierBytes = static_cast<size_t>(frontierCapacity) * sizeof(int32_t);
    const uint32_t wordsPerBitmap = (((n + 31u) >> 5) + 15u) / 16u * 16u;
    const size_t nextBitmapBytes = static_cast<size_t>(cfg.block_dim) * wordsPerBitmap * sizeof(uint32_t);

    try {
    netbfs::Timer setupTimer;
    ACL_CHECK(aclrtCreateStream(&stream));
    ACL_CHECK(aclrtMalloc(&dev.rowPtr, rowPtrBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&dev.colIdx, colIdxBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&dev.frontier, frontierBytes, ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMalloc(&dev.nextBitmap, nextBitmapBytes, ACL_MEM_MALLOC_HUGE_FIRST));

    ACL_CHECK(aclrtMemcpy(dev.rowPtr, rowPtrBytes, graph.row_ptr.data(), rowPtrBytes, ACL_MEMCPY_HOST_TO_DEVICE));
    ACL_CHECK(aclrtMemcpy(dev.colIdx, colIdxBytes, colIdxPadded.data(), colIdxBytes, ACL_MEMCPY_HOST_TO_DEVICE));

    std::cout << "static_graph_setup_us=" << setupTimer.elapsed_us() << "\n";
    for (uint32_t i = 0; i < cfg.warmup; ++i) {
        (void)run_device_bfs_once(cfg, graph, colIdxPadded, dev, stream, false);
    }

    netbfs::BfsResult last;
    netbfs::TimingUs avg;
    bool allPass = true;
    const uint32_t repeat = std::max(1u, cfg.repeat);
    for (uint32_t i = 0; i < repeat; ++i) {
        last = run_device_bfs_once(cfg, graph, colIdxPadded, dev, stream, true);
        const auto c = netbfs::compare_results(last, ref);
        allPass = allPass && c.replacement_ok && c.distance_ok && c.level_ok;
        avg.bfs_total_us += last.timing.bfs_total_us;
        avg.kernel_us += last.timing.kernel_us;
        avg.host_frontier_us += last.timing.host_frontier_us;
    }
    avg.bfs_total_us /= repeat;
    avg.kernel_us /= repeat;
    avg.host_frontier_us /= repeat;
    last.timing = avg;


##### （4）结果输出、CSV 写入与 Host 主函数

检查候选节点、距离和层号，写入 CSV；任一正式运行校验失败时返回非零值。


In [ ]:
%%writefile -a Source/02.05/ascend_ops/host_launch/bfs_npu_main.cpp
    const auto check = netbfs::compare_results(last, ref);
    const std::string status = allPass ? "PASS" : "FAIL";
    netbfs::print_result_row(graph, cfg.fault_node, last, status, true);
    const std::string method = cfg.sweep ? "NPU_BFS_L" + std::to_string(cfg.frontier_block_len) : "NPU_BFS";
    netbfs::write_result_csv(cfg.csv_path, method, avg.bfs_total_us, status, append);
    std::cout << "diagnosis: replacement=" << last.replacement_node
              << ", distance=" << last.distance << '\n';
    if (!allPass) std::cerr << "BFS validation failed; last level mismatches=" << check.level_mismatch << '\n';
    if (cfg.print_levels) netbfs::print_levels_sample(last);

    free_buffers(dev);
    aclrtDestroyStream(stream);
    return allPass;
    } catch (...) {
        if (stream) (void)aclrtSynchronizeStream(stream);
        free_buffers(dev);
        if (stream) (void)aclrtDestroyStream(stream);
        throw;
    }
}

}  // namespace

int main(int argc, char** argv) {
    bool initialized=false, deviceSet=false, argumentsValid=false;
    int device=0;
    try {
        const Config cfg = parse_args(argc, argv);
        netbfs::check_config(cfg);
        argumentsValid=true;

        // 在构图前检查设备；设备初始化不计入 BFS 执行时间。
        report_device_environment();
        ACL_CHECK(aclInit(nullptr)); initialized=true;
        device=cfg.device;
        validate_device_index(device);
        ACL_CHECK(aclrtSetDevice(device)); deviceSet=true;
        if (cfg.check_device_only) {
            ACL_CHECK(aclrtResetDevice(device)); deviceSet=false;
            ACL_CHECK(aclFinalize()); initialized=false;
            std::cout << "Device initialization PASSED\n";
            return 0;
        }

        netbfs::Timer prepTimer;
        const auto graph = netbfs::make_network_graph(cfg);
        const double prepUs = prepTimer.elapsed_us();
        for (uint32_t i = 0; i < cfg.warmup; ++i)
            (void)netbfs::serial_bfs_reference(graph, cfg.fault_node, netbfs::effective_max_depth(cfg));
        netbfs::BfsResult ref;
        double serialTotal = 0.0;
        for (uint32_t i = 0; i < cfg.repeat; ++i) {
            ref = netbfs::serial_bfs_reference(graph, cfg.fault_node, netbfs::effective_max_depth(cfg));
            serialTotal += ref.timing.bfs_total_us;
        }
        ref.timing.bfs_total_us = serialTotal / cfg.repeat;

        if (cfg.print_graph) netbfs::dump_csr_small(graph);
        netbfs::print_graph_summary(graph, cfg.fault_node);
        std::cout << "reference: replacement=" << ref.replacement_node
                  << ", distance=" << ref.distance
                  << ", visited=" << ref.visited_count
                  << ", processed_edges=" << ref.processed_edges
                  << ", preprocess_us=" << std::fixed << std::setprecision(2) << prepUs
                  << ", serial_bfs_us=" << ref.timing.bfs_total_us << std::defaultfloat << "\n";


        netbfs::print_result_header(true);
        bool allPass = true;
        bool append = false;
        if (cfg.sweep) {
            for (uint32_t len : {256u, 512u, 1024u, 2048u}) {
                Config cur = cfg;
                cur.frontier_block_len = len;
                allPass = run_one(cur, graph, ref, append) && allPass;
                append = true;
            }
        } else {
            allPass = run_one(cfg, graph, ref);
        }

        ACL_CHECK(aclrtResetDevice(device)); deviceSet=false;
        ACL_CHECK(aclFinalize()); initialized=false;
        return allPass ? 0 : 2;
    } catch (const std::exception& e) {
        if (deviceSet) (void)aclrtResetDevice(device);
        if (initialized) (void)aclFinalize();
        std::cerr << "error: " << e.what() << "\n";
        if (!argumentsValid) usage(argv[0]);
        return 1;
    }
}


#### 3.1.5 `Source/02.05/scripts`：构建、运行和 profiling 脚本

统一使用 `run.sh` 构建和运行，`profile.sh` 在已经构建的 NPU 程序上进行可选性能采集。


##### （1）完整构建与运行脚本：`run.sh`

复用已验证参考 Notebook 的 CANNLab 路径检测、型号检测和兼容目录处理；`--cpu` 模式仅构建 CPU。


In [ ]:
%%writefile Source/02.05/scripts/run.sh
#!/usr/bin/env bash
set -euo pipefail
# CANNLab environment adapter. Source this file, then call
# prepare_cann_environment <build-dir> [installation-hint] [soc-hint].
is_cann_install_path() {
  local path="$1"
  [[ -f "${path}/tikcpp/ascendc_kernel_cmake/ascendc.cmake" ||
     -f "${path}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake" ||
     -f "${path}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake" ]]
}

detect_cann_install_path() {
  local arch base candidate
  arch=$(uname -m)
  # 指定安装目录时只使用该目录，避免切换版本时回退到其他 CANN。
  if [[ -n "${1:-}" ]]; then
    for candidate in "${1}/${arch}-linux" "${1}"; do
      if is_cann_install_path "${candidate}"; then
        printf '%s\n' "${candidate}"
        return 0
      fi
    done
    return 1
  fi
  for base in "${ASCEND_INSTALL_PATH:-}" \
              "${ASCEND_HOME_PATH:-}" "${ASCEND_TOOLKIT_HOME:-}" \
              /opt/conda/Ascend/cann-* /usr/local/Ascend/ascend-toolkit/latest \
              /usr/local/Ascend/ascend-toolkit/*; do
    [[ -n "${base}" ]] || continue
    for candidate in "${base}/${arch}-linux" "${base}"; do
      if is_cann_install_path "${candidate}"; then
        printf '%s\n' "${candidate}"
        return 0
      fi
    done
  done
  return 1
}

detect_soc_version() {
  local raw_name normalized
  command -v npu-smi >/dev/null 2>&1 || return 1
  raw_name=$(npu-smi info 2>/dev/null \
    | sed -nE 's/^\|[[:space:]]*[0-9]+[[:space:]]+([^|]+)\|.*/\1/p' \
    | head -n 1 || true)
  normalized=$(printf '%s' "${raw_name}" | tr '[:upper:]' '[:lower:]' | tr -cd '[:alnum:]')
  case "${normalized}" in
    ascend910*|ascend310*) printf '%s\n' "${normalized}" ;;
    910*|310*) printf 'ascend%s\n' "${normalized}" ;;
    *) return 1 ;;
  esac
}

prepare_cann_environment() {
  local BUILD_DIR="$1" path_hint="${2:-}" soc_hint="${3:-${SOC_VERSION:-}}"
  local detected env_file="" candidate
  if [[ -n "${path_hint}" && ! -d "${path_hint}" ]]; then
    echo "CANN installation does not exist: ${path_hint}" >&2
    return 1
  fi
  if ! detected=$(detect_cann_install_path "${path_hint}"); then
    echo "Cannot find CANN Ascend C package. Set ASCEND_INSTALL_PATH or use -a <path>." >&2
    return 1
  fi
  ASCEND_INSTALL_PATH="${detected}"
  for candidate in "${ASCEND_INSTALL_PATH}/set_env.sh" "${ASCEND_INSTALL_PATH}/../set_env.sh"; do
    if [[ -f "${candidate}" ]]; then env_file="${candidate}"; break; fi
  done
  if [[ -n "${env_file}" ]]; then
    # Vendor environment scripts may read unset variables.
    set +u
    source "${env_file}"
    set -u
  fi
  # Keep the architecture-specific package selected before sourcing.
  ASCEND_INSTALL_PATH="${detected}"
  if [[ -n "${soc_hint}" ]]; then
    SOC_VERSION="${soc_hint}"
  elif detected_soc=$(detect_soc_version); then
    SOC_VERSION="${detected_soc}"
  else
    echo "Cannot detect SOC_VERSION with npu-smi. Set SOC_VERSION or use -v <soc>." >&2
    return 1
  fi
  mkdir -p "${BUILD_DIR}"
ASCEND_CMAKE_PACKAGE_PATH="${ASCEND_INSTALL_PATH}"
ASCENDC_CMAKE_NEW_LAYOUT="${ASCEND_INSTALL_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
BISHENG_REAL="${ASCEND_INSTALL_PATH}/ccec_compiler/bin/bisheng"
BISHENG_EXPECTED="${ASCEND_INSTALL_PATH}/ascendc_devkit/ccec_compiler/bin/bisheng"

if [[ -f "${ASCENDC_CMAKE_NEW_LAYOUT}" && -x "${BISHENG_REAL}" && ! -x "${BISHENG_EXPECTED}" ]]; then
  ASCEND_CMAKE_PACKAGE_PATH="${BUILD_DIR}/cann_package_compat"
  mkdir -p "${ASCEND_CMAKE_PACKAGE_PATH}/ascendc_devkit"

  for entry in "${ASCEND_INSTALL_PATH}"/*; do
    [[ -e "${entry}" ]] || continue
    name=$(basename "${entry}")
    [[ "${name}" == "ascendc_devkit" ]] && continue
    ln -sfn "${entry}" "${ASCEND_CMAKE_PACKAGE_PATH}/${name}"
  done

  if [[ -d "${ASCEND_INSTALL_PATH}/ascendc_devkit" ]]; then
    for entry in "${ASCEND_INSTALL_PATH}/ascendc_devkit"/*; do
      [[ -e "${entry}" ]] || continue
      name=$(basename "${entry}")
      [[ "${name}" == "ccec_compiler" ]] && continue
      ln -sfn "${entry}" "${ASCEND_CMAKE_PACKAGE_PATH}/ascendc_devkit/${name}"
    done
  fi

  for component in asc tikcpp ccec_compiler; do
    if [[ -e "${ASCEND_INSTALL_PATH}/${component}" ]]; then
      ln -sfn "${ASCEND_INSTALL_PATH}/${component}" \
        "${ASCEND_CMAKE_PACKAGE_PATH}/ascendc_devkit/${component}"
    fi
  done
  echo "[INFO] Using CANN package compatibility view: ${ASCEND_CMAKE_PACKAGE_PATH}"
fi

export ASCEND_INSTALL_PATH
  export ASCEND_CANN_PACKAGE_PATH="${ASCEND_CMAKE_PACKAGE_PATH}"
  export SOC_VERSION
  export LD_LIBRARY_PATH="${ASCEND_INSTALL_PATH}/lib64:${ASCEND_INSTALL_PATH}/runtime/lib64:${LD_LIBRARY_PATH:-}"
  echo "[INFO] ASCEND_INSTALL_PATH=${ASCEND_INSTALL_PATH}"
  echo "[INFO] SOC_VERSION=${SOC_VERSION}"
}

run_experiment_main() {
    local ROOT BUILD_DIR CANN_HINT="" SOC_HINT="${SOC_VERSION:-}" CLEAN=0
    ROOT=$(cd "$(dirname "${BASH_SOURCE[0]}")/.." && pwd)
    if [[ "${1:-}" == "--cpu" ]]; then
        shift
        BUILD_DIR="${ROOT}/build_cpu"
        cmake -S "${ROOT}" -B "${BUILD_DIR}" -DCMAKE_BUILD_TYPE=Release -DBUILD_ASCEND=OFF
        cmake --build "${BUILD_DIR}" -j
        cd "${ROOT}"
        mkdir -p results
        "${BUILD_DIR}/network_bfs_demo" "$@"
        return
    fi
    BUILD_DIR="${ROOT}/build_ascend"
    local OPTIND=1 opt
    while getopts ":a:v:ch" opt; do
        case "${opt}" in
            a) CANN_HINT="${OPTARG}" ;;
            v) SOC_HINT="${OPTARG}" ;;
            c) CLEAN=1 ;;
            h) echo "Usage: $0 [-a CANN_PATH] [-v SOC] [-c] [-- program options]";
               echo "CPU only: $0 --cpu [program options]"; return 0 ;;
            *) echo "Invalid option; use -h" >&2; return 1 ;;
        esac
    done
    shift $((OPTIND - 1))
    if [[ "${1:-}" == "--" ]]; then shift; fi
    if [[ "${CLEAN}" == 1 ]]; then rm -rf "${BUILD_DIR}"; fi
    prepare_cann_environment "${BUILD_DIR}" "${CANN_HINT}" "${SOC_HINT}"
    cmake -S "${ROOT}" -B "${BUILD_DIR}" -DCMAKE_BUILD_TYPE=Release \
        -DBUILD_ASCEND=ON -DRUN_MODE=npu -DSOC_VERSION="${SOC_VERSION}" \
        -DASCEND_CANN_PATH="${ASCEND_CANN_PACKAGE_PATH}" \
        -DASCEND_CANN_PACKAGE_PATH="${ASCEND_CANN_PACKAGE_PATH}"
    cmake --build "${BUILD_DIR}" -j
    cd "${ROOT}"
    mkdir -p results
    "${BUILD_DIR}/network_bfs_ascend_demo" "$@"
}
if [[ "${BASH_SOURCE[0]}" == "$0" ]]; then run_experiment_main "$@"; fi


##### （2）Profiling 脚本：`profile.sh`

复用环境检测并调用 msprof，将采集结果写入独立目录。


In [ ]:
%%writefile Source/02.05/scripts/profile.sh
#!/usr/bin/env bash
set -euo pipefail
ROOT=$(cd "$(dirname "${BASH_SOURCE[0]}")/.." && pwd)
source "${ROOT}/scripts/run.sh"
CANN_HINT=""
SOC_HINT="${SOC_VERSION:-}"
while getopts ":a:v:h" opt; do
    case "${opt}" in
        a) CANN_HINT="${OPTARG}" ;;
        v) SOC_HINT="${OPTARG}" ;;
        h) echo "Usage: $0 [-a CANN_PATH] [-v SOC] [-- program options]"; exit 0 ;;
        *) echo "Invalid option; use -h" >&2; exit 1 ;;
    esac
done
shift $((OPTIND - 1))
if [[ "${1:-}" == "--" ]]; then shift; fi
prepare_cann_environment "${ROOT}/build_ascend" "${CANN_HINT}" "${SOC_HINT}"
command -v msprof >/dev/null 2>&1 || { echo "msprof not found" >&2; exit 1; }
BIN="${ROOT}/build_ascend/network_bfs_ascend_demo"
[[ -x "${BIN}" ]] || { echo "Run scripts/run.sh first" >&2; exit 1; }
mkdir -p "${ROOT}/results/profile"
printf -v APPLICATION '%q ' "${BIN}" --csv "${ROOT}/results/profile_result.csv" "$@"
msprof --output="${ROOT}/results/profile" --application="${APPLICATION}"


### 3.2 CMake 构建

`CMakeLists.txt` 集中组织 CPU 公共库、CPU 程序、Ascend C 核函数库与 NPU Host 程序。`BUILD_ASCEND=OFF` 时可以独立构建 CPU 部分；NPU 脚本传入检测到的 CANN 路径和型号，并设置 `BUILD_ASCEND=ON`。

实际配置与编译由第 3.3 节脚本执行。


#### 3.2.1 CMake 配置文件：`CMakeLists.txt`

组织公共源码和启动接口，兼容参考环境中的 tikcpp 包目录。


In [ ]:
%%writefile Source/02.05/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(ascendc_parallel_bfs_diagnosis LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

if(NOT CMAKE_BUILD_TYPE)
  set(CMAKE_BUILD_TYPE Release CACHE STRING "Build type" FORCE)
endif()

option(BUILD_ASCEND "Build Ascend C NPU demo" OFF)

add_library(network_bfs_cpu
    Code/network_bfs_cpu.cpp
)
target_include_directories(network_bfs_cpu PUBLIC Code/include)
target_compile_options(network_bfs_cpu PRIVATE -Wall -Wextra -Wpedantic)

add_executable(network_bfs_demo Code/main.cpp)
target_link_libraries(network_bfs_demo PRIVATE network_bfs_cpu)
target_compile_options(network_bfs_demo PRIVATE -Wall -Wextra -Wpedantic)

if(BUILD_ASCEND)
  set(RUN_MODE "npu" CACHE STRING "Ascend C run mode: npu/cpu/sim")
  set(SOC_VERSION "$ENV{SOC_VERSION}" CACHE STRING "Ascend SOC version, e.g. ascend910b1/ascend910b2/ascend310p3")
  set(ASCEND_CANN_PATH "$ENV{ASCEND_INSTALL_PATH}" CACHE PATH "CANN installation path")
  if(NOT ASCEND_CANN_PATH OR NOT SOC_VERSION)
    message(FATAL_ERROR "Use scripts/run.sh to detect CANN and SOC_VERSION")
  endif()
  set(ASCEND_CANN_PACKAGE_PATH "${ASCEND_CANN_PATH}" CACHE PATH "CANN package path" FORCE)
  set(CMAKE_INSTALL_PREFIX "${CMAKE_BINARY_DIR}/out" CACHE PATH "Ascend C install output" FORCE)

  if(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  else()
    message(FATAL_ERROR "Cannot find ascendc.cmake under ${ASCEND_CANN_PACKAGE_PATH}. Check ASCEND_CANN_PATH/ASCEND_INSTALL_PATH.")
  endif()

  message(STATUS "ASCEND_CANN_PACKAGE_PATH=${ASCEND_CANN_PACKAGE_PATH}")
  message(STATUS "SOC_VERSION=${SOC_VERSION}")
  include("${ASCENDC_CMAKE_FILE}")

  ascendc_library(network_bfs_kernels STATIC
      ascend_ops/op_kernel/bfs_expand_level.cpp
  )
  ascendc_compile_definitions(network_bfs_kernels PRIVATE
      -DASCENDC_DUMP=0
  )

  add_executable(network_bfs_ascend_demo
      ascend_ops/host_launch/bfs_npu_main.cpp
  )
  target_include_directories(network_bfs_ascend_demo PRIVATE
      Code/include
      ${ASCEND_CANN_PACKAGE_PATH}/include
      ${ASCEND_CANN_PACKAGE_PATH}/include/external
      ${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp
      ${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/tikcfw
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/include
      ${CMAKE_INSTALL_PREFIX}/include/network_bfs_kernels
      ${CMAKE_BINARY_DIR}/out/include/network_bfs_kernels
  )
  target_link_directories(network_bfs_ascend_demo PRIVATE
      ${ASCEND_CANN_PACKAGE_PATH}/lib64
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64
      ${ASCEND_CANN_PACKAGE_PATH}/acllib/lib64
      ${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/devlib
      ${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/devlib
  )
  target_link_libraries(network_bfs_ascend_demo PRIVATE
      network_bfs_kernels
      network_bfs_cpu
      ascendcl platform ascendalog c_sec dl
  )
  add_dependencies(network_bfs_ascend_demo network_bfs_kernels)
endif()

install(TARGETS network_bfs_demo RUNTIME DESTINATION bin)
install(DIRECTORY ascend_ops DESTINATION share/ascendc_parallel_bfs_diagnosis)


#### 3.2.2 检查工程文件并设置脚本执行权限

检查源码位置并使用 `bash -n` 检查脚本语法。该步骤不代表 NPU 编译或运行已经通过。


In [ ]:
required = ['Code/include/network_bfs_common.h', 'Code/include/network_bfs_cpu.h',
            'Code/network_bfs_cpu.cpp', 'Code/main.cpp',
            'ascend_ops/op_kernel/bfs_expand_level.cpp',
            'ascend_ops/host_launch/bfs_npu_main.cpp', 'scripts/run.sh',
            'scripts/profile.sh', 'CMakeLists.txt']
for name in required:
    path = WORK_DIR / name
    assert path.is_file(), f"缺少文件：{path}"
    if path.suffix == '.sh':
        path.chmod(0o755)
        subprocess.run(['bash', '-n', str(path)], check=True)
    print(name)


### 3.3 运行并输出实验结果

先用小规模参数完成 CPU 与 NPU 运行检查，再调整输入规模。两种执行方式使用同一组参数。若修改参数，应重新运行这两个单元，以便比较同一图上的结果。

每次运行前清除对应旧 CSV，保存本轮配置和日志。NPU 运行包含逐层邻居扩展、结果回传和 Host 合并；构建或验证失败时，单元会明确报错。

NPU Host 在构图前先检查可见设备数量和设备初始化，失败时输出 AscendCL 的详细错误。`--device` 使用当前进程可见的设备索引；容器映射后，该索引可能与物理编号不同。可使用 `bash Source/02.05/scripts/run.sh -- --device 0 --check-device` 单独检查设备初始化。该检查不执行 BFS，也不生成性能结果。


In [ ]:
import json
import subprocess

NODES = 262144
AVG_DEGREE = 32
FAULT_NODE = 0
HEALTH_PERCENT = 1
FAULT_RADIUS = 4
FRONTIER_BLOCK_LEN = 32
NEIGHBOR_TILE_LEN = 32
BLOCK_DIM = 16
WARMUP, REPEAT = 2, 10
SEED = 1234
MAX_DEPTH = 0
DEVICE_ID = 0

def current_config():
    return dict(nodes=NODES, avg_degree=AVG_DEGREE, fault=FAULT_NODE,
                health=HEALTH_PERCENT, radius=FAULT_RADIUS,
                frontier_len=FRONTIER_BLOCK_LEN, neighbor_len=NEIGHBOR_TILE_LEN,
                blocks=BLOCK_DIM, warmup=WARMUP, repeat=REPEAT,
                seed=SEED, max_depth=MAX_DEPTH, device=DEVICE_ID)

def common_args():
    return ['--nodes', str(NODES), '--avg-degree', str(AVG_DEGREE),
            '--fault-node', str(FAULT_NODE), '--health-percent', str(HEALTH_PERCENT),
            '--fault-radius', str(FAULT_RADIUS), '--seed', str(SEED),
            '--max-depth', str(MAX_DEPTH), '--warmup', str(WARMUP), '--repeat', str(REPEAT)]

def run_experiment(kind, command):
    result_dir = WORK_DIR / 'results'
    csv_path = result_dir / f'bfs_{kind}.csv'
    config_path = result_dir / f'{kind}_config.json'
    csv_path.unlink(missing_ok=True)
    config_path.unlink(missing_ok=True)
    with (result_dir / f'{kind}.log').open('w', encoding='utf-8') as log:
        proc = subprocess.Popen(command, cwd=WORK_DIR, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
        returncode = proc.wait()
    if csv_path.exists():
        config_path.write_text(json.dumps(current_config(), indent=2), encoding='utf-8')
    if returncode:
        raise RuntimeError(f'{kind} 构建、运行或校验失败，请检查上方日志。')
    assert csv_path.is_file(), f'未生成结果：{csv_path}'

run_experiment('cpu', ['bash', 'scripts/run.sh', '--cpu'] + common_args()
               + ['--csv', 'results/bfs_cpu.csv'])


In [ ]:
# 本单元需要 CANN 与昇腾 NPU。
run_experiment('npu', ['bash', 'scripts/run.sh', '--'] + common_args()
               + ['--device', str(DEVICE_ID), '--block-dim', str(BLOCK_DIM),
                  '--frontier-block-len', str(FRONTIER_BLOCK_LEN),
                  '--neighbor-tile-len', str(NEIGHBOR_TILE_LEN),
                  '--csv', 'results/bfs_npu.csv'])


---
## 4. 正确性验证与性能分析

前面已经完成实验的构建、运行和结果输出，本节读取上一节生成的 `results/bfs_npu.csv`，对CPU与NPU的实验结果进行对比分析。

### 4.1 实验指标

**status**：

NPU 每次正式运行都与 CPU 参考比较替代节点、距离和全部 `level` 项。三者均相同时，`status` 为 `PASS`；任一次不一致则为 `FAIL`，程序返回非零值。CPU 行标记为 `REFERENCE`，表示它是校验依据。

**avg_us**：

`avg_us` 是正式运行的平均时间，单位为微秒。CPU 时间从初始访问状态准备完成后开始，覆盖逐层 BFS；NPU 时间从初始 Host 状态准备完成后开始，覆盖各层前沿传入、位图清零、Kernel 启动与同步、位图回传及 Host 合并与候选选择。链式构图、状态准备、CSR 转换和链表释放计入独立的预处理时间；两种 BFS 时间均不包含这些步骤。NPU 时间也不含首次设备内存申请与静态图传入、CPU 参考生成及最终结果比较。

### 4.2 CPU 参考与 NPU 结果对比


In [ ]:
import csv
import json
from IPython.display import display, Markdown

columns = ["method", "avg_us", "status"]
rows = []
for kind in ["cpu", "npu"]:
    csv_path = WORK_DIR / "results" / f"bfs_{kind}.csv"
    config_path = WORK_DIR / "results" / f"{kind}_config.json"
    if not csv_path.exists() or not config_path.exists():
        print(f"{kind} 结果未生成，请完成第 3.3 节对应运行。")
        continue
    if json.loads(config_path.read_text()) != current_config():
        print(f"{kind} 的参数与本轮配置不一致，请重新运行。")
        continue
    with csv_path.open(newline="", encoding="utf-8") as f:
        rows.extend(csv.DictReader(f))
if rows:
    table = ["| " + " | ".join(columns) + " |", "|" + "---|" * len(columns)]
    for row in rows:
        table.append("| " + " | ".join(row[k] if k in ["method", "status"]
            else f"{float(row[k]):.3f}" for k in columns) + " |")
    display(Markdown("\n".join(table)))
    with (WORK_DIR / "results/bfs_summary.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)
else:
    (WORK_DIR / "results/bfs_summary.csv").unlink(missing_ok=True)

matched = {row['method']: row for row in rows}
if 'CPU_BFS' in matched and 'NPU_BFS' in matched and matched['NPU_BFS']['status'] == 'PASS':
    cpu_us = float(matched['CPU_BFS']['avg_us'])
    npu_us = float(matched['NPU_BFS']['avg_us'])
    if cpu_us > 0 and npu_us > 0:
        display(Markdown(f"CPU/NPU 时间比：**{cpu_us / npu_us:.3f}**。"))


---
## 5. 实验总结

本实验从网络边生成邻接表，使用头插法连接 `ArcNode`，沿 `nextarc` 遍历并转换为 CSR。`data` 与顶点编号相同，`arcnum` 统计无向边，每条边保留两条邻接记录。转换结束后逐条释放链表结点。

CPU 串行与 NPU 并行 BFS 共同使用 CSR。NPU 扩展当前层，Host 合并位图、更新层号并选择最近一层中编号最小的可用节点。正确性检查候选、距离与全部层号；性能分析区分共同预处理、静态设备准备与 BFS 协同执行时间。
